In [1]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference
from langchain_openai import ChatOpenAI
import os
import json
from typing import  List
from langchain.schema import Document
from langchain.storage import LocalFileStore
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from qdrant_client.http import models
from langchain_deepseek import ChatDeepSeek
#models/gemini-2.5-flash-lite
#llm = ChatNVIDIA(model="meta/llama3-8b-instruct")
#from langchain_mistralai import ChatMistralAI

#llm = ChatMistralAI(model="devstral-small-2505",api_key="0ABvp5FEW3e7CjvjqtRfOoZ35Is7wxky")

llm = ChatOpenAI(model="gpt-4o-mini")
evaluator_llm = LangchainLLMWrapper(llm)

d:\Studia\AquaRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#[i for i in ChatNVIDIA().available_models if i.model_type=='chat' and i.supports_tools]

In [2]:
import sys
sys.path.append(os.path.abspath(".."))

In [4]:
from backend.vector_db.parent_embedd import create_parent_retriever
from backend.vector_db.basic_embedd import create_retriver


retriever = create_parent_retriever()
#assert len(list(retriever.vectorstore.store.yield_keys())) > 0

🔍 Creating ParentDocumentRetriever...


In [5]:
def get_child_elements(keys_list: List[str]) -> List[str]:
    doc_list = []
    response, _ = retriever.vectorstore.client.scroll(
		collection_name=os.environ["COLLECTION_NAME"],
		scroll_filter=models.Filter(
			must=[
				models.HasIdCondition(has_id=keys_list)  # 💡 filtr po ID
			],
		),
		with_payload=True,
		with_vectors=False
	)
    doc_list = [i.payload['page_content'] for i in response][:3]
    return doc_list

In [13]:
def get_parent_elements(keys_list: List[str], fs: LocalFileStore) -> List[Document]:
    doc_list = []
    for key in keys_list:
        value = fs.mget([key])[0]
        decoded = value.decode("utf-8")
        data = json.loads(decoded)
        kwargs = data.get("kwargs")
        doc = Document(**kwargs)
        doc_list.append(doc)

    return doc_list

In [22]:
len_list = []
for i in retriever.docstore.yield_keys():
    element = get_parent_elements([i], retriever.docstore.store)
    len_list.append(len(element[0].page_content))

In [23]:
sum(len_list) / len(len_list)

2027.5

In [25]:
folder_path = "../output/multi_cross_parent"

json_obj_list = []
for file_path in os.listdir(folder_path):
    with open(folder_path + "/" + file_path, mode="r", encoding="UTF-8") as f:
        data = json.load(f)
        json_obj_list.append(data)

In [10]:
json_obj_list[0]

{'question': 'Na czym polegają melioracje wodne z artykułu 195?',
 'system_answer': 'Melioracje wodne, zgodnie z artykułem 195, polegają na regulacji stosunków wodnych w celu polepszenia zdolności produkcyjnej gleby i ułatwienia jej uprawy.',
 'parent_ids': ['652ac8c0-6fb8-48c8-9c45-30981f145465'],
 'child_ids': [],
 'retrival_tool': 'article',
 'id': 1,
 'reference': 'Melioracje wodne polegają na regulacji stosunków wodnych wcelu polepszenia zdolności produkcyjnej gleby i ułatwienia jej uprawy'}

### All llm metrics in one

In [17]:
from ragas.metrics import LLMContextPrecisionWithReference
from ragas.metrics import ContextEntityRecall
from ragas.metrics import LLMContextRecall
from ragas.metrics import NoiseSensitivity
from ragas.metrics import SemanticSimilarity
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from ragas.metrics import Faithfulness
from ragas.metrics import ResponseRelevancy
from ragas.metrics._factual_correctness import FactualCorrectness

evaluator_embedding = NVIDIAEmbeddings(model=os.environ["EMBEDDER"])

In [19]:
scorers = {
	#"context_precision_without_reference": LLMContextPrecisionWithoutReference(llm=evaluator_llm), 
	#"context_precision_with_reference": LLMContextPrecisionWithReference(llm=evaluator_llm),
	#'context_recall': LLMContextRecall(llm=evaluator_llm),
	#'context_entity_recall': ContextEntityRecall(llm=evaluator_llm),
	'context_noise_sensitivity': NoiseSensitivity(llm=evaluator_llm),
	'context_semantic_similarity': SemanticSimilarity(embeddings=LangchainEmbeddingsWrapper(evaluator_embedding)),
	'faithfulness': Faithfulness(llm=evaluator_llm),
	'context_response_relevancy': ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embedding),
	'context_factual_correctness': FactualCorrectness(llm = evaluator_llm)
}
scores_dict = {
	#"context_precision_without_reference": [], 
	#"context_precision_with_reference": [],
	#'context_recall': [],
	#'context_entity_recall': [],
	'context_noise_sensitivity': [],
	'context_semantic_similarity': [],
	'faithfulness': [],
	'context_response_relevancy': [],
	'context_factual_correctness': []
}
j = 0
for key, scorer in scorers.items():
	scores = []
	print("Calculating key... " , key)
	for item in json_obj_list:
		retrieved_contexts = [
			i for i in get_child_elements(item['child_ids'][:3])
		]
		sample = SingleTurnSample(
			user_input=item['question'],
			response=item['system_answer'],
			reference=item['reference'],
			retrieved_contexts=retrieved_contexts 
		)
	
		score = await scorer.single_turn_ascore(sample)
		scores.append(score)

	print(key, scores)
	scores_dict[key] = scores

Calculating key...  context_noise_sensitivity
context_noise_sensitivity [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.6666666666666666), np.float64(0.0), np.float64(0.0), np.float64(0.25), np.float64(0.0), np.float64(0.45454545454545453), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.1), np.float64(0.0), np.float64(0.0), np.float64(0.1111111111111111), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.2857142857142857), np.float64(0.2), np.float64(0.6363636363636364), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
Calculating key...  context_semantic_similarity
context_semantic_similarity [0.9551371724032256, 0.6269865297318846, 0.6779988308420619, 0.7111110785869738, 0.6324040566752316, 0.8088462857227817, 0.9120283676164291, 0.6851951984538818, 0.9439914497876625, 0.9107065333115465, 0.9126540379703945, 0.9726482838294608, 0.9799404848247635, 0.9

In [11]:
folder_path = "../output/cross_parent"

json_obj_list = []
for file_path in os.listdir(folder_path):
    with open(folder_path + "/" + file_path, mode="r", encoding="UTF-8") as f:
        data = json.load(f)
        json_obj_list.append(data)

In [12]:
import spacy
from rapidfuzz.distance import Levenshtein, Hamming, Jaro, JaroWinkler

# załaduj polski model spaCy (upewnij się, że masz go zainstalowanego: python -m spacy download pl_core_news_sm)
nlp = spacy.load("pl_core_news_sm")

def normalize_text(text: str) -> str:
    doc = nlp(text.lower())
    tokens = [
        token.lemma_   # lematyzacja po polsku
        for token in doc
        if not token.is_stop and token.is_alpha  # usuwamy stopwords i znaki nie-alfabetyczne
    ]
    return " ".join(tokens)

lev_scores, ham_scores, jaro_scores, jw_scores = [], [], [], []

for item in json_obj_list:
    reference = normalize_text(item['reference'])
    system_answer = normalize_text(item['system_answer'])
    
    # Levenshtein
    lev_score = Levenshtein.normalized_similarity(reference, system_answer)
    lev_scores.append(lev_score)
    
    # Hamming (dopasowanie długości paddingiem spacjami)
    max_len = max(len(reference), len(system_answer))
    ref_padded = reference.ljust(max_len)
    sys_padded = system_answer.ljust(max_len)
    ham_score = Hamming.normalized_similarity(ref_padded, sys_padded)
    ham_scores.append(ham_score)
    
    # Jaro
    jaro_score = Jaro.similarity(reference, system_answer)
    jaro_scores.append(jaro_score)
    
    # Jaro-Winkler
    jw_score = JaroWinkler.similarity(reference, system_answer)
    jw_scores.append(jw_score)

# średnie wyniki
mean_lev = sum(lev_scores) / len(lev_scores)
mean_ham = sum(ham_scores) / len(ham_scores)
mean_jaro = sum(jaro_scores) / len(jaro_scores)
mean_jw = sum(jw_scores) / len(jw_scores)

print("Średnie wyniki metryk AquaRAG-no-upgrades:")
print(f"Levenshtein: {mean_lev:.4f}")
print(f"Hamming:     {mean_ham:.4f}")
print(f"Jaro:        {mean_jaro:.4f}")
print(f"JaroWinkler: {mean_jw:.4f}")


Średnie wyniki metryk AquaRAG-no-upgrades:
Levenshtein: 0.5048
Hamming:     0.1894
Jaro:        0.7414
JaroWinkler: 0.7707


In [13]:
import pandas as pd
df_scores = pd.DataFrame({
    'lev_scores': lev_scores,
    'ham_scores': ham_scores,
    'jaro_scores': jaro_scores,
    'jw_scores': jw_scores
})

# now each list is a column, and each element is a row
print(df_scores)

    lev_scores  ham_scores  jaro_scores  jw_scores
0     0.859504    0.190083     0.822619   0.893572
1     0.137809    0.106007     0.531624   0.531624
2     0.221053    0.089474     0.605168   0.605168
3     0.480663    0.082873     0.681294   0.681294
4     0.274648    0.079812     0.672828   0.672828
5     0.536364    0.109091     0.688623   0.688623
6     0.743961    0.188406     0.819972   0.891983
7     0.348592    0.169014     0.728643   0.837186
8     0.641369    0.093750     0.738854   0.843312
9     0.774752    0.319307     0.839541   0.903725
10    0.366120    0.131148     0.737745   0.842647
11    0.866197    0.848592     0.948624   0.969174
12    0.316794    0.148855     0.681505   0.681505
13    0.702179    0.215496     0.788801   0.873281
14    0.318841    0.061594     0.767752   0.767752
15    0.349265    0.132353     0.738441   0.843064
16    0.684492    0.200535     0.817926   0.890756
17    0.838710    0.034946     0.815012   0.833510
18    0.723077    0.038462     

In [14]:
df_scores.to_csv('../output/final_results/nlp/cross_parent_distances.csv')

### NLP metrics

In [83]:
from nltk.translate.bleu_score import sentence_bleu

weights = (0.25, 0, 0, 0)  # Weights for uni-gram, bi-gram, tri-gram, and 4-gram

# Reference and predicted texts (same as before)
reference = [["Jacek", "je", "dużo", "słodyczy", "bo", "bardzo", "lubi", "czekoladę", "i", "lody" ]]
predictions = ["Jacek", "je", "dużo", "kałamarnic", "bo", "bardzo", "lubi", "czekoladę", "i", "lody" ]


bleu = []
for item in json_obj_list:
    retrieved_contexts = [
        i.page_content for i in get_parent_elements(item['parent_ids'][:3], retriever.docstore.store)
    ]
    reference = [item['reference'].split()]
    prediction = item['system_answer'].split()
    score = sentence_bleu(reference, predictions, weights=weights)
    bleu.append(score)

print(bleu)

[0.3086194627209984, 0.5623413251903491, 0.12547530994780284, 0, 0, 0, 0, 0, 0.0051146501783410625, 0.0030204197170134897, 0, 0, 0.030941830482760235, 0.013903060638709325, 0.01138286330681933, 0.018767168855425027, 0, 0, 0, 0.18718716632432567, 0, 0.01138286330681933, 0, 6.939843277158474e-05, 0, 0.5623413251903491, 0.3086194627209984, 0, 0, 0]


d:\Studia\AquaRAG\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
d:\Studia\AquaRAG\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
d:\Studia\AquaRAG\.venv\Lib\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.

In [84]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL', 'rouge2'], use_stemmer=True)

scores = []
for item in json_obj_list:
    retrieved_contexts = [
        i for i in get_child_elements(item['child_ids'])
    ]

    score = scorer.score(target=item['reference'],prediction=item['system_answer'])
    scores.append(score)

print(scores)

[{'rouge1': Score(precision=0.72, recall=0.9473684210526315, fmeasure=0.8181818181818181), 'rougeL': Score(precision=0.72, recall=0.9473684210526315, fmeasure=0.8181818181818181), 'rouge2': Score(precision=0.625, recall=0.8333333333333334, fmeasure=0.7142857142857143)}, {'rouge1': Score(precision=0.375, recall=1.0, fmeasure=0.5454545454545454), 'rougeL': Score(precision=0.375, recall=1.0, fmeasure=0.5454545454545454), 'rouge2': Score(precision=0.34782608695652173, recall=1.0, fmeasure=0.5161290322580645)}, {'rouge1': Score(precision=0.13157894736842105, recall=0.3125, fmeasure=0.18518518518518517), 'rougeL': Score(precision=0.10526315789473684, recall=0.25, fmeasure=0.14814814814814814), 'rouge2': Score(precision=0.013333333333333334, recall=0.03225806451612903, fmeasure=0.018867924528301886)}, {'rouge1': Score(precision=0.5142857142857142, recall=1.0, fmeasure=0.6792452830188679), 'rougeL': Score(precision=0.5142857142857142, recall=1.0, fmeasure=0.6792452830188679), 'rouge2': Score(p

In [85]:
rouge1_precision = []
rouge1_recall = []
rouge1_f1 = []

rouge2_precision = []
rouge2_recall = []
rouge2_f1 = []

rougel_precision = []
rougel_recall = []
rougel_f1 = []
for rouge in scores:
	rouge1_precision.append(rouge['rouge1'].precision)
	rouge1_recall.append(rouge['rouge1'].recall)
	rouge1_f1.append(rouge['rouge1'].fmeasure)
	rouge2_precision.append(rouge['rouge2'].precision)
	rouge2_recall.append(rouge['rouge2'].recall)
	rouge2_f1.append(rouge['rouge2'].fmeasure)
	rougel_precision.append(rouge['rougeL'].precision)
	rougel_recall.append(rouge['rougeL'].recall)
	rougel_f1.append(rouge['rougeL'].fmeasure)

In [86]:
scores_dict.update({
    'rouge1_precision': rouge1_precision,
    'rouge1_recall': rouge1_recall,
    'rouge1_f1': rouge1_f1,
    'rouge2_precision': rouge2_precision,
    'rouge2_recall': rouge2_recall,
    'rouge2_f1': rouge2_f1,
    'rougel_precision': rougel_precision,
    'rougel_recall': rougel_recall,
    'rougel_f1': rougel_f1,
    'bleu': bleu
})

In [28]:
import werpy

input_data = ["Melioracje wodne, zgodnie z artykułem 195, polegają na regulacji stosunków wodnych w celu polepszenia zdolności produkcyjnej gleby i ułatwienia jej uprawy.",
              "Melioracje wodne polegają na regulacji stosunków wodnych w celu polepszenia zdolności produkcyjnej gleby i ułatwienia jej uprawy"]
reference = werpy.normalize(input_data)
werpy.wer(reference[0], reference[1])

0.19047619047619047

In [ ]:
import werpy
refernce = []
system_answer = []
for item in json_obj_list:
	input_data = []
	refernce.append(werpy.normalize(item['reference']))
	system_answer.append(werpy.normalize(item['system_answer']))

werp = werpy.werp(refernce, system_answer, insertions_weight=0, deletions_weight=0, substitutions_weight=1)
print(werp)

0.5044483985765125
